# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sujithauday/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

For [SEO / content team], deciding [Which of the pages are most worth fixing?], we will build [the ranked pages to work on first using probabilty score] from [data], predicting [Yes/No label] measured by [precision at a threshold and precision@k]. A wrong call costs [not able to spend on much needed pages]. A plain rule isn't enough because [rule can not be applied to all scenarios it will only make model only memorize rules, data is huge and need to look for patterns that work best even in most similar scenarios]. We will claim only [decision-support to help with ranking pages to work on] results.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv")
print(df.head(3))

             content_id          client_id  search_volume  competition  \
0  content_304f48230142  client_f369cb89fc           10.0         0.67   
1  content_a1fb4e703a9e  client_4e07408562           90.0         0.01   
2  content_9aa793d4d895  client_7f2253d7e2            0.0         0.00   

  competition_level   cpc     content_type    main_intent  word_count  \
0              HIGH  2.05  keyword article  transactional      3221.0   
1               LOW  0.05  keyword article  informational      2481.0   
2               LOW  0.00  keyword article  informational      3515.0   

   char_count  ... char_count_tier   ctr  avg_position  engagement_rate  \
0     20457.0  ...     15000-25000  0.76          10.6             5.88   
1     15562.0  ...     15000-25000  0.05          20.3             0.00   
2     23643.0  ...     15000-25000  0.09          36.5             0.00   

   scroll_rate  ai_traffic_pct  impression_tier  position_tier  \
0         4.55             0.0             

My lane CTR / Engagement Opportunity Scoring
    Find which pages need fix first and rank top 50 of them?
    worth_to_fix - yes / no
    one row is one webpage
    
Good - showing pages whose fix truly increases their visibility, clicks and engagement./ Amoung the worth_to_fix pages how many were actually true.
Bad - Doing changes has no impact on the pages and has other factors to check and needs more detail work. Predicting worth_to_fix for nagative pages.

My section is supervised
- classify each row as worth_to_fix - This will short list pages who are already in right step but need a few tweaks and changes can easily improve their growth
- probability score for each page above treshold will be treated as Yes worth_fixing
- based on scores rank the Yes worth_fixing pages



## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

* Binary Classification: Decide if the declining page is worth a fix first. A Yes/ No label (worth_to_fix) from an observed outcome. Based of threshold limit. What limit will it be.
* Ranking / scoring: We need some scoring to rank the pages. A priority score.

Model output:  Predict which of the declining pages needs work first.
Ideal outcome: Rank the pages to work on first.

Observed outcome: having high traffic / search volume but Reduce in clicks and engagement in the prev_30 day and las_30 day. Not using percentage  

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Search volume is it worth a feature to look at for a rule

n_null = df['search_volume'].isnull()
print('Rows with no value for search_volume: ',n_null.sum())
print(df.loc[n_null, 'trend_direction'].value_counts())

nonull_rows = df[df['search_volume'].notna()]
q = nonull_rows['search_volume'].quantile([0.25, 0.50, 0.75])

for p, val in q.items():
    subset = df[df['search_volume'] == val]
    print(f"{p*100}% percentile trend_direction:")
    print(subset['trend_direction'].value_counts())

Rows with no value for search_volume:  2468
trend_direction
new       1012
down       737
up         310
stable     278
flat       131
Name: count, dtype: int64
25.0% percentile trend_direction:
trend_direction
down      7008
stable    1997
up        1430
new        344
flat       302
Name: count, dtype: int64
50.0% percentile trend_direction:
trend_direction
down      3850
stable    1633
up        1117
new        423
flat       288
Name: count, dtype: int64
75.0% percentile trend_direction:
trend_direction
down      1181
stable     516
up         348
new        146
flat        99
Name: count, dtype: int64


So as search_volume quantile is distributed across down trend_direction, its hard to defining a rule with high demad keywords will be.

In [4]:

# A page is eligible for a label only if content_age_days ≥ some minimum (e
# df['freshness_tier']

floors = [(0,0),(1,999), (1000,2999),(3000,4999),(5000,7400)]          # e.g. 0, 10, or 20 — put your decision here
for low, high in floors:
    eligible = (df['search_volume'] >= low) &(df['search_volume'] <= high)  # NaN > anything is False, so NaNs auto-excluded

    print(f"Final floor check: search_volume > {low} - {high}")
    print(f"  Eligible rows: {eligible.sum()} ({eligible.sum()/len(df)*100:.1f}%)")

eligible = (df['search_volume'].isna())
print(f"Final floor check: search_volume is NAN")
print(f"  Eligible rows: {eligible.sum()} ({eligible.sum()/len(df)*100:.1f}%)")


Final floor check: search_volume > 0 - 0
  Eligible rows: 11081 (36.9%)
Final floor check: search_volume > 1 - 999
  Eligible rows: 15787 (52.6%)
Final floor check: search_volume > 1000 - 2999
  Eligible rows: 430 (1.4%)
Final floor check: search_volume > 3000 - 4999
  Eligible rows: 82 (0.3%)
Final floor check: search_volume > 5000 - 7400
  Eligible rows: 53 (0.2%)
Final floor check: search_volume is NAN
  Eligible rows: 2468 (8.2%)


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Find ctr change for similar pages actual CTR change next week Last 10 similar pages improved by __%

* Classification: precision Vs base rate
* Ranking: Metric - is precision at 50

Base rate is of all eligible pages what fraction is actually worth_to_fix first.

Good: Around some percentage of the worth_to_fix output pages by the model are actually right.

Do you know your base rate yet — roughly what fraction of the 30k rows would qualify as "worth_fixing" once you define the label? If not, that's the very next thing to check, and it's real EDA, not a guess.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show that 30,000k rows has unique content_id showing that clients have different webpages

one row = one webpage. Find out of these 50 pages how many of them truly needed it. Model output - A score for each page and sorted in a rank queue Action - SEO gets the 50 pages worth fixing on.

find the features. output based on what

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# a simple rule based on ctr/position/volume alone can't distinguish "this page is new and hasn't ranked yet" from "this page is old and has been declining" —
# ("high volume + good position + low ctr") missing column values

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.